# Gold Layer: Patient 360 Analytical Model

# Overview
This notebook combines active records across all Silver tables to build a business-ready **Patient 360** star schema view.

# Aggregated Metrics
**total_encounters**: Distinct count of patient encounters.
**total_observations**: Distinct count of clinical observations recorded.
**total_conditions**: Distinct count of diagnosed conditions.

In [0]:
# Create Gold Analytical Delta Table aggregating patient records across current dimensions
gold_sql = """
CREATE OR REPLACE TABLE gold_patient_360 AS
SELECT 
    p.patient_id,
    p.gender,
    p.birth_date,
    COUNT(DISTINCT e.encounter_id) AS total_encounters,
    COUNT(DISTINCT o.observation_id) AS total_observations,
    COUNT(DISTINCT c.condition_id) AS total_conditions,
    MAX(p.effective_start_date) AS last_updated
FROM silver_patient p
LEFT JOIN silver_encounter e 
    ON CONCAT('Patient/', p.patient_id) = e.patient_ref AND e.is_current = true
LEFT JOIN silver_observation o 
    ON CONCAT('Patient/', p.patient_id) = o.patient_ref AND o.is_current = true
LEFT JOIN silver_condition c 
    ON CONCAT('Patient/', p.patient_id) = c.patient_ref AND c.is_current = true
WHERE p.is_current = true
GROUP BY p.patient_id, p.gender, p.birth_date
"""

spark.sql(gold_sql)
print("--- Gold Layer Table 'gold_patient_360' Created Successfully ---")

# Display sample output for review
display(spark.sql("SELECT * FROM gold_patient_360 LIMIT 10"))

--- Gold Layer Table 'gold_patient_360' Created Successfully ---


patient_id,gender,birth_date,total_encounters,total_observations,total_conditions,last_updated
74a69fd5-3c9c-51c5-8d58-0a3d1f769171,male,1985-09-20,0,0,0,2026-07-23T06:01:01.503Z
burnout-demo-patient-kowalski-2026-07-05-0,null,null,0,0,0,2026-07-23T06:01:04.529Z
burnout-demo-patient-kowalski-2026-04-15-0,null,null,0,0,0,2026-07-23T06:01:04.529Z
23e615fa-0068-46df-b064-10bb8a827e7f,female,1985-05-15,0,0,0,2026-07-23T06:01:04.529Z
burnout-demo-patient-almeida-2026-04-23-0,null,null,0,0,0,2026-07-23T06:01:06.481Z
137215536,male,1980-01-01,0,0,2,2026-07-23T06:01:01.503Z
9ce3ac58-385f-4f32-805a-f79081e9b8cb,female,1978-11-12,0,0,0,2026-07-23T06:01:04.529Z
137215109,female,1962-04-12,0,0,0,2026-07-23T06:01:01.503Z
burnout-demo-patient-almeida-2026-05-09-0,null,null,0,0,0,2026-07-23T06:01:06.481Z
burnout-demo-patient-patel-2026-06-16-1,null,null,0,0,0,2026-07-23T06:01:06.481Z
